# M34 lab solution: Personal RAG

**Brief.** Give your assistant semantic memory of user preferences and show that answers in a new thread use them.

In a first thread the user, Priya, states six preferences and facts (`../data/preferences.jsonl`). A LangGraph graph extracts each one and stores it in an `InMemoryStore` with an embedding index, as in `04a_episodic_with_semantic_memory.ipynb`. In a second thread the same graph answers four handbook questions, recalling the memories closest to each question. A run for a user with no memories answers the same questions for comparison.

The notebook reads only `../data/` and imports only from `src/`. Run it from this folder; the lab README lists the keys it needs.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
sys.path.insert(0, str(Path("../../../../src").resolve()))

from helpers import get_embeddings, get_experientiallabs_llm
from lab_kit import is_hit, load_handbook, load_jsonl

DATA = Path("../data")
llm = get_experientiallabs_llm()
print("LLM:", llm.model_name)

## The Part V system, rebuilt inline

This lab does not depend on another lab's output, so the cell below rebuilds the minimal Part V system: the handbook loaded with `lab_kit`, the baseline chunker (500 characters, 50 overlap), an in-memory vector store over `get_embeddings()`, and an answer prompt that cites passages by number. Later cells reuse `retriever`, `format_docs` and `rag_answer`.

In [ ]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

handbook = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [Document(page_content=c, metadata={"source": p["source"]}) for p in handbook for c in splitter.split_text(p["text"])]
embeddings = get_embeddings()
store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about the Larkspur Cloud handbook. Use only the numbered passages. "
               "Cite each claim with its passage number in square brackets, like [2]. "
               "If the passages do not contain the answer, reply: The handbook does not say."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])
answer_chain = ANSWER_PROMPT | llm | StrOutputParser()


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata['source']}) {d.page_content}" for i, d in enumerate(docs, 1))


def rag_answer(question, docs=None):
    """Retrieve (unless docs are given) and answer with citations; returns (answer, docs)."""
    docs = retriever.invoke(question) if docs is None else docs
    return answer_chain.invoke({"context": format_docs(docs), "question": question}), docs


print(len(chunks), "chunks;", len(golden), "golden questions")

## 1. A store with an embedding index

`index=` makes the store embed the `text` field of every value on `put()`, so `search(namespace, query=...)` ranks memories by meaning.

In [ ]:
from dataclasses import dataclass

from langgraph.store.memory import InMemoryStore

dims = len(embeddings.embed_query("dimension probe"))
memory_store = InMemoryStore(index={"embed": embeddings, "dims": dims, "fields": ["text"]})


@dataclass
class Context:
    user_id: str  # trusted: set from the login, never by the model


def prefs_ns(user_id):
    return (user_id, "preferences")


print("Embedding dimensions:", dims)

## 2. The graph: remember, recall, answer

One graph serves both threads. A turn that states a preference is stored; a turn that asks a question recalls the three closest memories and answers from them and the handbook passages.

In [ ]:
from typing import Literal, Optional, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime
from pydantic import BaseModel, Field


class Memory(BaseModel):
    topic: str = Field(description="A one- or two-word topic used as the memory's key, such as region, plan or answer style")
    text: str = Field(description="The preference or fact, as a short third-person sentence")


class Extracted(BaseModel):
    kind: Literal["preference_or_fact", "question"]
    memories: list[Memory] = Field(default_factory=list)


extractor = ChatPromptTemplate.from_messages([
    ("system", "Classify the user's message. If it states preferences or facts about the user or their account, "
               "list each one with a short topic and a third-person sentence. If it asks a question, return no memories."),
    ("human", "{message}"),
]) | llm.with_structured_output(Extracted)

PERSONAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about the Larkspur Cloud handbook from the numbered passages only. "
               "What you know about this user:\n{memories}\n"
               "Apply the user's preferences and account details when they are relevant."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])
personal_chain = PERSONAL_PROMPT | llm | StrOutputParser()


class TurnState(TypedDict):
    message: str
    kind: str
    recalled: list[str]
    reply: str


def classify(state: TurnState, runtime: Runtime[Context]):
    out = extractor.invoke({"message": state["message"]})
    for m in out.memories:  # semantic memory: upsert under a stable key, so a changed preference replaces the old one
        runtime.store.put(prefs_ns(runtime.context.user_id), m.topic.strip().lower(), {"text": m.text})
    return {"kind": out.kind, "reply": "Noted: " + "; ".join(m.text for m in out.memories) if out.memories else ""}


def answer(state: TurnState, runtime: Runtime[Context]):
    recalled = [it.value["text"] for it in runtime.store.search(prefs_ns(runtime.context.user_id),
                                                                query=state["message"], limit=3)]
    docs = retriever.invoke(state["message"])
    reply = personal_chain.invoke({"memories": "\n".join(f"- {m}" for m in recalled) or "- nothing yet",
                                   "context": format_docs(docs), "question": state["message"]})
    return {"recalled": recalled, "reply": reply}


builder = StateGraph(TurnState, context_schema=Context)
builder.add_node("classify", classify)
builder.add_node("answer", answer)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", lambda s: "answer" if s["kind"] == "question" else END, ["answer", END])
builder.add_edge("answer", END)
assistant = builder.compile(checkpointer=InMemorySaver(), store=memory_store)


def turn(message, user_id, thread_id):
    return assistant.invoke({"message": message, "kind": "", "recalled": [], "reply": ""},
                            {"configurable": {"thread_id": thread_id}}, context=Context(user_id=user_id))

## 3. The first thread: Priya states her preferences

In [ ]:
for p in load_jsonl(DATA / "preferences.jsonl"):
    out = turn(p["text"], user_id=p["user_id"], thread_id="priya-first")
    print(f"[{p['states']}] {p['text']}\n    -> {out['reply']}")
stored = memory_store.search(prefs_ns("priya"), limit=50)
print(f"\n{len(stored)} memories stored for priya")

## 4. A second thread, with and without memory

The new thread shares no messages with the first; only the store connects them. `nobody` is a user with no memories, so the same graph answers without personal context. The expected word is what an answer that used the stored preference would contain.

In [ ]:
SECOND_THREAD = [
    ("Which region should I create our next server in?", "Frankfurt"),
    ("How long are our daily backups kept?", "7 days"),
    ("How do I get a copy of last year's invoices?", "finance"),
    ("How quickly will support answer our tickets?", "4 hours"),
]
rows = []
for question, expected in SECOND_THREAD:
    with_memory = turn(question, user_id="priya", thread_id="priya-second")
    without = turn(question, user_id="nobody", thread_id="nobody-second")
    rows.append({"question": question, "recalled": " | ".join(with_memory["recalled"]),
                 "with memory": with_memory["reply"], "without memory": without["reply"],
                 f"expected word": expected,
                 "used (with)": expected.lower() in with_memory["reply"].lower(),
                 "used (without)": expected.lower() in without["reply"].lower()})
side_by_side = pd.DataFrame(rows)
for _, r in side_by_side.iterrows():
    print(f"Q: {r.question}\n  recalled: {r.recalled}\n  WITH MEMORY:    {r['with memory'][:300]!r}\n"
          f"  WITHOUT MEMORY: {r['without memory'][:300]!r}\n")

## 5. Results

In [ ]:
print(side_by_side[["question", "expected word", "used (with)", "used (without)"]].to_string(index=False))
print(f"\nAnswers that used a stored preference: {side_by_side['used (with)'].sum()} of {len(side_by_side)} with memory, "
      f"{side_by_side['used (without)'].sum()} of {len(side_by_side)} without")

## What this shows

- The side-by-side answers are the brief's evidence: the second thread starts with no messages, yet the answers with memory can name Frankfurt, the Business plan's backup and response times, and the finance team, because the store returned those memories for the question.
- The expected-word check is a crude test, chosen because it needs no judge. An answer without memory can still contain the word (both answers may list every region); read the answers, not just the flags.
- Recall is by meaning, at three memories per question. A question whose relevant memory is not in its top three answers as if the memory did not exist; print `recalled` for every miss before changing `limit`.